# Tableau 1 + contrôle qualité du recalage + versions des librairies

Exécute *Exécution → Tout exécuter*. Le notebook produit dans `nemo_results/figures_article/` :
- `table1.csv` et `table1.tex` : caractéristiques des 127 patients, au total et par groupe, avec les p-valeurs ;
- `qc_registration.txt` : distribution du rapport de volume natif/MNI et volumes lésionnels ;
- `versions.txt` : versions des librairies.

**Envoie-moi ensuite ces 3 fichiers.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torch_geometric shap

In [ ]:
import os, re, glob, numpy as np, pandas as pd, torch
import torch_geometric.typing
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact
BASE = "/content/drive/MyDrive/isles_prp/MNI"
NEMO = os.path.join(BASE, "nemo_results")
OUT  = os.path.join(NEMO, "figures_article"); os.makedirs(OUT, exist_ok=True)

ds = torch.load(os.path.join(NEMO, "pyg_dataset.pt"), weights_only=False)
coh = pd.DataFrame({"pid": [d.pid.lower() for d in ds], "y": [int(d.y) for d in ds],
                    "SumChaCo": [float(d.x[:, 0].sum()) for d in ds]})
rows = []
for pid in coh.pid:
    f = glob.glob(os.path.join(BASE, pid, "*demographic_baseline.csv"))
    r = pd.read_csv(f[0], encoding="utf-8-sig").iloc[0].to_dict() if f else {}
    r = {k.strip(): v for k, v in r.items()}; r["pid"] = pid; rows.append(r)
clin = pd.DataFrame(rows)
df = coh.merge(clin, on="pid", how="left")
df["Male"] = (df["Sex"].astype(str).str.upper().str.strip() == "M").astype(float)
df.loc[df["Sex"].isna(), "Male"] = np.nan
log = os.path.join(BASE, "_recalage_log.csv")
if os.path.exists(log):
    lg = pd.read_csv(log); lg["pid"] = lg["patient"].str.lower()
    lg = lg[lg["statut"] == "OK"].drop_duplicates("pid", keep="last")
    df = df.merge(lg[["pid", "vol_natif_cm3", "vol_mni_cm3", "ratio"]], on="pid", how="left")
print(len(df), "patients | favorable:", (df.y == 0).sum(), "| unfavorable:", (df.y == 1).sum())

In [ ]:
CONT = [("Age, years", "Age"), ("NIHSS at admission", "NIHSS at admission"),
        ("Premorbid mRS", "mRS premorbid"), ("Glucose", "Glucose"),
        ("Leukocytes", "Leucocytes"), ("CRP", "CRP"), ("INR", "INR"),
        ("Lesion volume (MNI), mL", "vol_mni_cm3"), ("Total disconnection ($\\sum$ChaCo)", "SumChaCo")]
BIN  = [("Male sex", "Male"), ("Atrial fibrillation", "Atrial fibrillation"), ("Hypertension", "Hypertension"),
        ("Diabetes", "Diabetes"), ("Hyperlipidemia", "Hyperlipidemia"), ("Prior anticoagulation", "Anticoagulation"),
        ("Prior lipid-lowering drugs", "Lipid lowering drugs"), ("Prior antiplatelet agents", "PAIs"),
        ("Wake-up stroke", "Wake-up"), ("In-hospital stroke", "In-House"), ("Referral", "Referral")]
g0, g1 = df[df.y == 0], df[df.y == 1]
def fp(p): return "--" if p != p else ("$<$0.001" if p < 0.001 else f"{p:.3f}")
def cont(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0: return "--"
    q1, m, q3 = np.percentile(s, [25, 50, 75])
    d = 0 if s.abs().max() >= 100 else (1 if s.abs().max() >= 10 else 2)
    return f"{m:.{d}f} [{q1:.{d}f}--{q3:.{d}f}]"
def binv(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return f"{int(s.sum())} ({100 * s.mean():.0f}\\%)" if len(s) else "--"
out = []
for lab, c in CONT:
    if c not in df: continue
    a, b = pd.to_numeric(g0[c], errors="coerce").dropna(), pd.to_numeric(g1[c], errors="coerce").dropna()
    p = mannwhitneyu(a, b).pvalue if len(a) and len(b) else np.nan
    miss = int(pd.to_numeric(df[c], errors="coerce").isna().sum())
    out.append([lab + " (median [IQR])", cont(df[c]), cont(g0[c]), cont(g1[c]), fp(p), miss])
for lab, c in BIN:
    if c not in df: continue
    s = pd.to_numeric(df[c], errors="coerce"); ok = s.notna()
    tab = pd.crosstab(df.y[ok], s[ok])
    if tab.shape == (2, 2):
        p = fisher_exact(tab)[1] if (tab.values < 5).any() else chi2_contingency(tab)[1]
    else: p = np.nan
    out.append([lab + ", $n$ (\\%)", binv(df[c]), binv(g0[c]), binv(g1[c]), fp(p), int((~ok).sum())])
T = pd.DataFrame(out, columns=["Characteristic", "All", "mRS<=2", "mRS>2", "p", "missing"])
T.to_csv(os.path.join(OUT, "table1.csv"), index=False)
n, n0, n1 = len(df), len(g0), len(g1)
tex = ["\\begin{tabular*}{\\textwidth}{@{\\extracolsep\\fill}lcccc}", "\\toprule",
       f"Characteristic & All ($n={n}$) & mRS $\\leq 2$ ($n={n0}$) & mRS $> 2$ ($n={n1}$) & $p$ \\\\", "\\midrule"]
tex += [f"{r[0]} & {r[1]} & {r[2]} & {r[3]} & {r[4]} \\\\" for r in out]
tex += ["\\botrule", "\\end{tabular*}"]
open(os.path.join(OUT, "table1.tex"), "w").write("\n".join(tex))
pd.set_option("display.width", 200); print(T.to_string(index=False))

In [ ]:
txt = []
if "ratio" in df:
    r = pd.to_numeric(df["ratio"], errors="coerce").dropna()
    q = np.percentile(r, [0, 25, 50, 75, 100])
    txt.append(f"Patients with QC log: {len(r)}/{len(df)}")
    txt.append(f"Volume ratio MNI/native: median {q[2]:.2f} [IQR {q[1]:.2f}-{q[3]:.2f}], range {q[0]:.2f}-{q[4]:.2f}")
    for lo, hi in [(0.8, 1.2), (0.7, 1.3), (0.5, 1.5)]:
        txt.append(f"  within [{lo}, {hi}]: {int(((r >= lo) & (r <= hi)).sum())}/{len(r)}")
    out_r = df.loc[pd.to_numeric(df["ratio"], errors="coerce").sub(1).abs() > 0.3, ["pid", "ratio", "vol_natif_cm3", "vol_mni_cm3"]]
    txt.append("Patients with ratio outside [0.7, 1.3]:\n" + (out_r.to_string(index=False) if len(out_r) else "  none"))
    v = pd.to_numeric(df["vol_mni_cm3"], errors="coerce").dropna()
    txt.append(f"Lesion volume (MNI): median {v.median():.1f} mL [IQR {v.quantile(.25):.1f}-{v.quantile(.75):.1f}], range {v.min():.1f}-{v.max():.1f}")
else:
    txt.append("_recalage_log.csv not found in " + BASE)
open(os.path.join(OUT, "qc_registration.txt"), "w").write("\n".join(txt)); print("\n".join(txt))

In [ ]:
import importlib, sys
vers = [f"python {sys.version.split()[0]}"]
for m in ["torch", "torch_geometric", "sklearn", "xgboost", "shap", "networkx", "nilearn", "numpy", "scipy", "pandas"]:
    try: vers.append(f"{m} {importlib.import_module(m).__version__}")
    except Exception: vers.append(f"{m} (not installed in this session)")
open(os.path.join(OUT, "versions.txt"), "w").write("\n".join(vers)); print("\n".join(vers))